# Smart Contract Risk Assessment Demo

This notebook demonstrates the comprehensive smart contract security and interaction risk assessment engine for Algorand protocols.

## Features Demonstrated:
- **Contract Security Analysis**: Comprehensive security assessment including audit status, vulnerabilities, and code quality
- **Interaction Risk Assessment**: Analysis of contract interaction patterns and associated risks
- **Exploit Proximity Analysis**: Distance from known exploits and vulnerability patterns
- **Admin Key Risk Assessment**: Centralization and governance risks
- **Dependency Analysis**: Contract dependency chains and single points of failure

In [ ]:
# Import required libraries
import asyncio
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import networkx as nx
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

# Set up plotting
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")
%matplotlib inline

In [ ]:
# Import smart contract risk assessment modules
import sys
sys.path.append('../')

from smart_contract_risk.core.contract_engine import SmartContractRiskEngine
from smart_contract_risk.core.contract_analyzer import SmartContractAnalyzer
from smart_contract_risk.core.interaction_risk import InteractionRiskAnalyzer

## 1. Initialize Smart Contract Risk Engine

In [ ]:
# Initialize the smart contract risk engine
contract_engine = SmartContractRiskEngine()

print("Smart Contract Risk Assessment Engine initialized successfully!")
print(f"Configuration loaded with {len(contract_engine.config.get('contract_categories', {}))} contract categories")
print(f"Audit database includes {len(contract_engine.config.get('audit_database', {}).get('known_auditors', {}))} known auditors")

## 2. Sample Contract Portfolio

Let's define sample contracts representing different security profiles in the Algorand ecosystem.

In [ ]:
# Define sample contracts with different security profiles

sample_contracts = {
    # High-security audited protocol
    'algofi_pool': {
        'name': 'AlgoFi Lending Pool',
        'address': '465818260',
        'functions': ['deposit', 'withdraw', 'borrow', 'repay', 'liquidate', 'external_call'],
        'features': [
            'access_control', 'reentrancy_protection', 'emergency_pause', 
            'timelock_mechanism', 'input_validation', 'event_logging'
        ],
        'deployment_date': '2021-11-01',
        'lines_of_code': 1200,
        'code_coverage': 0.88,
        'test_coverage': 0.85,
        'audits': [
            {
                'auditor': 'runtime_verification',
                'date': '2021-12-01',
                'score': 92,
                'type': 'formal_verification',
                'vulnerabilities_found': 2,
                'vulnerabilities_fixed': 2
            },
            {
                'auditor': 'certik',
                'date': '2022-06-01',
                'score': 88,
                'type': 'security_audit',
                'vulnerabilities_found': 3,
                'vulnerabilities_fixed': 3
            }
        ]
    },
    
    # Medium-security DEX contract
    'tinyman_dex': {
        'name': 'Tinyman DEX Pool',
        'address': '552635992',
        'functions': ['swap', 'add_liquidity', 'remove_liquidity', 'external_call'],
        'features': ['access_control', 'slippage_protection', 'input_validation'],
        'deployment_date': '2021-10-01',
        'lines_of_code': 800,
        'code_coverage': 0.75,
        'test_coverage': 0.70,
        'audits': [
            {
                'auditor': 'halborn',
                'date': '2021-11-15',
                'score': 82,
                'type': 'security_audit',
                'vulnerabilities_found': 4,
                'vulnerabilities_fixed': 3
            }
        ]
    },
    
    # Oracle contract with specific risks
    'price_oracle': {
        'name': 'Algorand Price Oracle',
        'address': '789123456',
        'functions': ['update_price', 'get_price', 'validate_data', 'admin_update'],
        'features': ['access_control', 'staleness_check', 'deviation_check'],
        'deployment_date': '2022-01-01',
        'lines_of_code': 450,
        'code_coverage': 0.80,
        'test_coverage': 0.75,
        'audits': [
            {
                'auditor': 'trail_of_bits',
                'date': '2022-02-01',
                'score': 85,
                'type': 'security_audit',
                'vulnerabilities_found': 2,
                'vulnerabilities_fixed': 2
            }
        ]
    },
    
    # Bridge contract with higher risks
    'bridge_contract': {
        'name': 'Cross-Chain Bridge',
        'address': '987654321',
        'functions': ['lock', 'unlock', 'validate_proof', 'delegated_call', 'admin_withdraw'],
        'features': ['access_control', 'multisig_requirement'],  # Missing many security features
        'deployment_date': '2022-08-01',
        'lines_of_code': 1800,
        'code_coverage': 0.60,
        'test_coverage': 0.55,
        'audits': [
            {
                'auditor': 'immunefi',
                'date': '2022-09-01',
                'score': 68,
                'type': 'bug_bounty',
                'vulnerabilities_found': 8,
                'vulnerabilities_fixed': 5
            }
        ]
    },
    
    # Unaudited governance contract
    'governance_contract': {
        'name': 'Protocol Governance',
        'address': '123789456',
        'functions': ['propose', 'vote', 'execute', 'delegate', 'admin_override'],
        'features': ['access_control'],  # Minimal security features
        'deployment_date': '2023-03-01',
        'lines_of_code': 950,
        'code_coverage': 0.45,
        'test_coverage': 0.40
        # No audits - high risk
    }
}

print("Sample contract portfolio defined:")
for contract_id, contract_data in sample_contracts.items():
    audit_count = len(contract_data.get('audits', []))
    print(f"  {contract_data['name']}: {contract_data['lines_of_code']} LOC, {audit_count} audit(s)")

## 3. Contract Interaction History

Let's define realistic interaction patterns between these contracts.

In [ ]:
# Define contract interaction history
interaction_history = [
    # High-frequency oracle queries
    {
        'source': 'algofi_pool',
        'target': 'price_oracle',
        'type': 'price_query',
        'frequency': 200,
        'gas_usage': 25000,
        'success_rate': 0.99,
        'last_interaction': '2024-01-15T10:30:00'
    },
    {
        'source': 'tinyman_dex',
        'target': 'price_oracle',
        'type': 'price_query',
        'frequency': 180,
        'gas_usage': 22000,
        'success_rate': 0.98,
        'last_interaction': '2024-01-15T10:25:00'
    },
    
    # Governance interactions
    {
        'source': 'governance_contract',
        'target': 'algofi_pool',
        'type': 'parameter_update',
        'frequency': 5,
        'gas_usage': 75000,
        'success_rate': 0.95,
        'last_interaction': '2024-01-10T14:20:00'
    },
    {
        'source': 'governance_contract',
        'target': 'price_oracle',
        'type': 'parameter_update',
        'frequency': 3,
        'gas_usage': 60000,
        'success_rate': 0.90,
        'last_interaction': '2024-01-08T16:45:00'
    },
    
    # Bridge interactions
    {
        'source': 'bridge_contract',
        'target': 'price_oracle',
        'type': 'validation_query',
        'frequency': 50,
        'gas_usage': 45000,
        'success_rate': 0.92,
        'last_interaction': '2024-01-14T08:15:00'
    },
    
    # Cross-DEX arbitrage pattern
    {
        'source': 'tinyman_dex',
        'target': 'algofi_pool',
        'type': 'token_swap',
        'frequency': 25,
        'gas_usage': 85000,
        'success_rate': 0.96,
        'last_interaction': '2024-01-14T12:30:00'
    },
    
    # Risky delegated call from bridge
    {
        'source': 'bridge_contract',
        'target': 'algofi_pool',
        'type': 'delegated_call',
        'frequency': 15,
        'gas_usage': 120000,
        'success_rate': 0.88,
        'last_interaction': '2024-01-12T18:45:00'
    }
]

print(f"Contract interaction history defined: {len(interaction_history)} interaction patterns")
print("\nInteraction Summary:")
for interaction in interaction_history:
    print(f"  {interaction['source']} → {interaction['target']}: {interaction['type']} ({interaction['frequency']} times)")

## 4. Comprehensive Contract Risk Assessment

Let's perform comprehensive risk assessment for the entire contract portfolio.

In [ ]:
# Perform comprehensive contract portfolio risk assessment
print("Performing comprehensive contract portfolio risk assessment...")

assessment = await contract_engine.assess_contract_portfolio_risk(
    sample_contracts, 
    interaction_history,
    include_exploit_analysis=True
)

print("\n=== Assessment Complete ===")
print(f"Overall Portfolio Risk: {assessment.overall_portfolio_risk:.2f}")
print(f"Risk Level: {assessment.risk_level}")
print(f"Confidence Score: {assessment.confidence_score:.2f}")
print(f"Next Review Date: {assessment.next_review_date.strftime('%Y-%m-%d')}")

print(f"\nPortfolio Metrics:")
metrics = assessment.portfolio_metrics
print(f"  Total Contracts: {metrics.total_contracts}")
print(f"  Audited Contracts: {metrics.audited_contracts}")
print(f"  High Risk Contracts: {metrics.high_risk_contracts}")
print(f"  Critical Vulnerabilities: {metrics.critical_vulnerabilities}")
print(f"  Complex Interactions: {metrics.complex_interactions}")

## 5. Individual Contract Security Analysis

Let's examine the security analysis for each contract in detail.

In [ ]:
# Analyze individual contract security scores
print("=== Individual Contract Security Analysis ===")

contract_scores_data = []

for score in assessment.contract_scores:
    print(f"\n{score.contract_name} ({score.contract_category}):")
    print(f"  Overall Risk Score: {score.overall_risk_score:.2f} ({score.risk_tier})")
    print(f"  Security Score: {score.security_score:.2f}")
    print(f"  Audit Score: {score.audit_score:.2f}")
    print(f"  Complexity Score: {score.complexity_score:.2f}")
    print(f"  Vulnerability Score: {score.vulnerability_score:.2f}")
    print(f"  Interaction Risk: {score.interaction_risk_score:.2f}")
    print(f"  Confidence Level: {score.confidence_level:.2f}")
    
    # Store data for visualization
    contract_scores_data.append({
        'Contract': score.contract_name,
        'Category': score.contract_category,
        'Overall Risk': score.overall_risk_score,
        'Security Score': score.security_score,
        'Audit Score': score.audit_score,
        'Complexity Risk': score.complexity_score,
        'Vulnerability Risk': score.vulnerability_score,
        'Interaction Risk': score.interaction_risk_score,
        'Risk Tier': score.risk_tier
    })

# Create DataFrame for analysis
scores_df = pd.DataFrame(contract_scores_data)
print("\n=== Contract Scores Summary ===")
print(scores_df.round(3))

In [ ]:
# Visualize contract security scores
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
fig.suptitle('Smart Contract Security Assessment Dashboard', fontsize=16, fontweight='bold')

# 1. Overall Risk Scores
ax1 = axes[0, 0]
colors = ['green' if tier == 'MINIMAL' else 'orange' if tier == 'LOW' else 'yellow' if tier == 'MODERATE' 
          else 'red' if tier == 'HIGH' else 'darkred' for tier in scores_df['Risk Tier']]
bars1 = ax1.bar(range(len(scores_df)), scores_df['Overall Risk'], color=colors, alpha=0.7)
ax1.set_title('Overall Contract Risk Scores')
ax1.set_ylabel('Risk Score (0-1)')
ax1.set_xticks(range(len(scores_df)))
ax1.set_xticklabels([name.split()[0] for name in scores_df['Contract']], rotation=45)
ax1.set_ylim(0, 1)

# Add risk tier labels
for i, (bar, tier) in enumerate(zip(bars1, scores_df['Risk Tier'])):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02, 
             tier, ha='center', va='bottom', fontsize=8, fontweight='bold')

# 2. Risk Component Breakdown
ax2 = axes[0, 1]
risk_components = ['Security Score', 'Audit Score', 'Complexity Risk', 'Vulnerability Risk', 'Interaction Risk']
x = np.arange(len(scores_df))
width = 0.15

for i, component in enumerate(risk_components):
    values = scores_df[component] if 'Risk' in component else 1 - scores_df[component]  # Invert scores for risk
    ax2.bar(x + i*width, values, width, label=component.replace(' Score', '').replace(' Risk', ''), alpha=0.7)

ax2.set_title('Risk Component Breakdown')
ax2.set_ylabel('Risk Level (0-1)')
ax2.set_xticks(x + width * 2)
ax2.set_xticklabels([name.split()[0] for name in scores_df['Contract']])
ax2.legend(bbox_to_anchor=(1.05, 1), loc='upper left')

# 3. Security vs Complexity
ax3 = axes[1, 0]
scatter = ax3.scatter(scores_df['Security Score'], scores_df['Complexity Risk'], 
                     s=scores_df['Overall Risk']*500, alpha=0.6, 
                     c=scores_df['Overall Risk'], cmap='RdYlGn_r')
ax3.set_xlabel('Security Score (Higher = Better)')
ax3.set_ylabel('Complexity Risk (Higher = Riskier)')
ax3.set_title('Security vs Complexity\n(Size = Overall Risk)')

# Add contract labels
for i, name in enumerate(scores_df['Contract']):
    ax3.annotate(name.split()[0], (scores_df['Security Score'].iloc[i], scores_df['Complexity Risk'].iloc[i]),
                xytext=(5, 5), textcoords='offset points', fontsize=8)

# 4. Risk Distribution by Category
ax4 = axes[1, 1]
category_risk = scores_df.groupby('Category')['Overall Risk'].mean().sort_values(ascending=False)
bars4 = ax4.bar(category_risk.index, category_risk.values, 
                color=['red', 'orange', 'yellow', 'lightblue', 'green'][:len(category_risk)], alpha=0.7)
ax4.set_title('Average Risk by Contract Category')
ax4.set_ylabel('Average Risk Score')
ax4.tick_params(axis='x', rotation=45)

for bar, risk in zip(bars4, category_risk.values):
    ax4.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01, 
             f'{risk:.2f}', ha='center', va='bottom', fontweight='bold')

plt.tight_layout()
plt.show()

## 6. Contract Interaction Risk Analysis

Let's analyze the interaction patterns and associated risks.

In [ ]:
# Analyze contract interactions
interaction_assessment = assessment.interaction_assessment

print("=== Contract Interaction Risk Analysis ===")
print(f"Overall Interaction Risk: {interaction_assessment.overall_interaction_risk:.2f}")
print(f"Interaction Risk Level: {interaction_assessment.risk_level}")

print(f"\nCall Graph Analysis:")
call_graph = interaction_assessment.call_graph
print(f"  Total Contracts: {len(call_graph.nodes)}")
print(f"  Total Interactions: {len(call_graph.edges)}")
print(f"  Max Call Depth: {call_graph.max_call_depth}")
print(f"  Circular Dependencies: {len(call_graph.circular_dependencies)}")
print(f"  Critical Paths: {len(call_graph.critical_paths)}")
print(f"  Complexity Score: {call_graph.complexity_score:.2f}")

print(f"\nInteraction Patterns Detected ({len(interaction_assessment.interaction_patterns)} total):")
for pattern in interaction_assessment.interaction_patterns:
    print(f"  {pattern.pattern_type} ({pattern.risk_level}):")
    print(f"    Description: {pattern.pattern_description}")
    print(f"    Contracts Involved: {len(pattern.contracts_involved)}")
    print(f"    Frequency: {pattern.interaction_frequency}")
    if pattern.vulnerability_indicators:
        print(f"    Vulnerabilities: {', '.join(pattern.vulnerability_indicators[:2])}")

print(f"\nCritical Interactions ({len(interaction_assessment.critical_interactions)} total):")
for critical in interaction_assessment.critical_interactions[:5]:  # Show top 5
    print(f"  ⚠️ {critical}")

In [ ]:
# Visualize contract interaction network
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 8))
fig.suptitle('Contract Interaction Network Analysis', fontsize=16, fontweight='bold')

# 1. Contract Interaction Graph
G = nx.DiGraph()

# Add nodes with risk-based coloring
for score in assessment.contract_scores:
    G.add_node(score.contract_name.split()[0], risk=score.overall_risk_score)

# Add edges from interactions
for interaction in interaction_assessment.contract_interactions:
    source = next((score.contract_name.split()[0] for score in assessment.contract_scores 
                  if score.contract_id == interaction.source_contract), interaction.source_contract)
    target = next((score.contract_name.split()[0] for score in assessment.contract_scores 
                  if score.contract_id == interaction.target_contract), interaction.target_contract)
    
    if source in G.nodes and target in G.nodes:
        G.add_edge(source, target, weight=interaction.risk_score, frequency=interaction.frequency)

# Layout and draw
pos = nx.spring_layout(G, seed=42)

# Node colors based on risk
node_colors = [G.nodes[node]['risk'] for node in G.nodes()]
node_sizes = [G.nodes[node]['risk'] * 1000 + 300 for node in G.nodes()]

# Draw network
nx.draw(G, pos, ax=ax1, with_labels=True, node_color=node_colors, 
        node_size=node_sizes, cmap='RdYlGn_r', font_size=8, font_weight='bold',
        edge_color='gray', arrows=True, arrowsize=20, alpha=0.7)
ax1.set_title('Contract Interaction Network\n(Size & Color = Risk Level)')

# 2. Interaction Risk Heatmap
# Create interaction risk matrix
contracts = [score.contract_name.split()[0] for score in assessment.contract_scores]
risk_matrix = np.zeros((len(contracts), len(contracts)))

for interaction in interaction_assessment.contract_interactions:
    source_idx = next((i for i, score in enumerate(assessment.contract_scores) 
                      if score.contract_id == interaction.source_contract), -1)
    target_idx = next((i for i, score in enumerate(assessment.contract_scores) 
                      if score.contract_id == interaction.target_contract), -1)
    
    if source_idx >= 0 and target_idx >= 0:
        risk_matrix[source_idx, target_idx] = interaction.risk_score

# Plot heatmap
im = ax2.imshow(risk_matrix, cmap='Reds', aspect='auto')
ax2.set_xticks(range(len(contracts)))
ax2.set_yticks(range(len(contracts)))
ax2.set_xticklabels(contracts, rotation=45, ha='right')
ax2.set_yticklabels(contracts)
ax2.set_title('Interaction Risk Matrix\n(Darker = Higher Risk)')

# Add text annotations
for i in range(len(contracts)):
    for j in range(len(contracts)):
        if risk_matrix[i, j] > 0:
            text = ax2.text(j, i, f'{risk_matrix[i, j]:.2f}', 
                           ha="center", va="center", color="white" if risk_matrix[i, j] > 0.5 else "black",
                           fontsize=8, fontweight='bold')

plt.tight_layout()
plt.show()

## 7. Exploit Proximity and Vulnerability Analysis

Let's examine the exploit proximity analysis and vulnerability patterns.

In [ ]:
# Analyze exploit proximity
exploit_proximity = assessment.exploit_proximity_analysis
upgrade_risks = assessment.upgrade_risk_analysis
admin_key_risks = assessment.admin_key_analysis

print("=== Exploit Proximity Analysis ===")
for contract_id, proximity in exploit_proximity.items():
    contract_name = next((score.contract_name for score in assessment.contract_scores 
                         if score.contract_id == contract_id), contract_id)
    risk_level = "HIGH" if proximity > 0.7 else "MEDIUM" if proximity > 0.4 else "LOW"
    emoji = "🔴" if risk_level == "HIGH" else "🟡" if risk_level == "MEDIUM" else "🟢"
    print(f"  {emoji} {contract_name}: {proximity:.2f} ({risk_level})")

print(f"\n=== Upgrade Risk Analysis ===")
for contract_id, upgrade_risk in upgrade_risks.items():
    contract_name = next((score.contract_name for score in assessment.contract_scores 
                         if score.contract_id == contract_id), contract_id)
    risk_level = "HIGH" if upgrade_risk > 0.7 else "MEDIUM" if upgrade_risk > 0.4 else "LOW"
    emoji = "🔴" if risk_level == "HIGH" else "🟡" if risk_level == "MEDIUM" else "🟢"
    print(f"  {emoji} {contract_name}: {upgrade_risk:.2f} ({risk_level})")

print(f"\n=== Admin Key Risk Analysis ===")
for contract_id, admin_risk in admin_key_risks.items():
    contract_name = next((score.contract_name for score in assessment.contract_scores 
                         if score.contract_id == contract_id), contract_id)
    risk_level = "HIGH" if admin_risk > 0.7 else "MEDIUM" if admin_risk > 0.4 else "LOW"
    emoji = "🔴" if risk_level == "HIGH" else "🟡" if risk_level == "MEDIUM" else "🟢"
    print(f"  {emoji} {contract_name}: {admin_risk:.2f} ({risk_level})")

In [ ]:
# Visualize comprehensive risk analysis
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
fig.suptitle('Comprehensive Contract Risk Analysis', fontsize=16, fontweight='bold')

# Prepare data for visualization
contract_names = [score.contract_name.split()[0] for score in assessment.contract_scores]
exploit_prox_values = [exploit_proximity.get(score.contract_id, 0) for score in assessment.contract_scores]
upgrade_risk_values = [upgrade_risks.get(score.contract_id, 0) for score in assessment.contract_scores]
admin_risk_values = [admin_key_risks.get(score.contract_id, 0) for score in assessment.contract_scores]
overall_risks = [score.overall_risk_score for score in assessment.contract_scores]

# 1. Exploit Proximity
ax1 = axes[0, 0]
colors1 = ['red' if x > 0.7 else 'orange' if x > 0.4 else 'green' for x in exploit_prox_values]
bars1 = ax1.bar(contract_names, exploit_prox_values, color=colors1, alpha=0.7)
ax1.set_title('Exploit Proximity Risk')
ax1.set_ylabel('Proximity Score (0-1)')
ax1.tick_params(axis='x', rotation=45)
ax1.set_ylim(0, 1)

# 2. Upgrade Risk
ax2 = axes[0, 1]
colors2 = ['red' if x > 0.7 else 'orange' if x > 0.4 else 'green' for x in upgrade_risk_values]
bars2 = ax2.bar(contract_names, upgrade_risk_values, color=colors2, alpha=0.7)
ax2.set_title('Upgrade Risk')
ax2.set_ylabel('Risk Score (0-1)')
ax2.tick_params(axis='x', rotation=45)
ax2.set_ylim(0, 1)

# 3. Admin Key Risk
ax3 = axes[1, 0]
colors3 = ['red' if x > 0.7 else 'orange' if x > 0.4 else 'green' for x in admin_risk_values]
bars3 = ax3.bar(contract_names, admin_risk_values, color=colors3, alpha=0.7)
ax3.set_title('Admin Key Centralization Risk')
ax3.set_ylabel('Risk Score (0-1)')
ax3.tick_params(axis='x', rotation=45)
ax3.set_ylim(0, 1)

# 4. Risk Correlation Analysis
ax4 = axes[1, 1]
risk_data = pd.DataFrame({
    'Overall Risk': overall_risks,
    'Exploit Proximity': exploit_prox_values,
    'Upgrade Risk': upgrade_risk_values,
    'Admin Risk': admin_risk_values
})

correlation_matrix = risk_data.corr()
im = ax4.imshow(correlation_matrix, cmap='RdBu_r', aspect='auto', vmin=-1, vmax=1)
ax4.set_xticks(range(len(correlation_matrix.columns)))
ax4.set_yticks(range(len(correlation_matrix.index)))
ax4.set_xticklabels(correlation_matrix.columns, rotation=45, ha='right')
ax4.set_yticklabels(correlation_matrix.index)
ax4.set_title('Risk Factor Correlations')

# Add correlation values
for i in range(len(correlation_matrix.index)):
    for j in range(len(correlation_matrix.columns)):
        text = ax4.text(j, i, f'{correlation_matrix.iloc[i, j]:.2f}', 
                       ha="center", va="center", color="white" if abs(correlation_matrix.iloc[i, j]) > 0.5 else "black",
                       fontweight='bold')

plt.tight_layout()
plt.show()

## 8. Risk Alerts and Recommendations

Let's examine the generated risk alerts and recommendations.

In [ ]:
# Display risk alerts
print("=== Smart Contract Risk Alerts ===")
print(f"Total Alerts: {len(assessment.risk_alerts)}")

if assessment.risk_alerts:
    # Sort alerts by urgency
    sorted_alerts = sorted(assessment.risk_alerts, key=lambda x: x.urgency_level, reverse=True)
    
    for i, alert in enumerate(sorted_alerts[:8], 1):  # Show top 8 alerts
        severity_emoji = {
            'CRITICAL': '🚨',
            'HIGH': '🔴',
            'MEDIUM': '🟡',
            'LOW': '🟢'
        }.get(alert.severity, '⚪')
        
        urgency_stars = '⭐' * min(alert.urgency_level, 5)
        
        print(f"\n{i}. {severity_emoji} {alert.severity} - {alert.alert_type.upper()}")
        print(f"   {urgency_stars} Urgency: {alert.urgency_level}/10")
        print(f"   📝 {alert.message}")
        print(f"   🎯 Action: {alert.recommended_action}")
        
        if alert.auto_mitigation_available:
            print(f"   🤖 Auto-mitigation available")
        
        if alert.contract_id:
            contract_name = next((score.contract_name for score in assessment.contract_scores 
                                 if score.contract_id == alert.contract_id), alert.contract_id)
            print(f"   📍 Contract: {contract_name}")
else:
    print("✅ No critical alerts detected - portfolio is in good shape!")

# Display recommendations
print("\n=== Risk Management Recommendations ===")
print(f"Total Recommendations: {len(assessment.recommendations)}")

if assessment.recommendations:
    for i, rec in enumerate(assessment.recommendations, 1):
        priority = "🔥" if "URGENT" in rec else "⚡" if "HIGH" in rec else "📋"
        print(f"  {i}. {priority} {rec}")
else:
    print("✅ No specific recommendations - continue current practices!")

# Display monitoring requirements
print("\n=== Monitoring Requirements ===")
print(f"Total Requirements: {len(assessment.monitoring_requirements)}")

if assessment.monitoring_requirements:
    for i, req in enumerate(assessment.monitoring_requirements, 1):
        print(f"  {i}. 🔍 {req}")
else:
    print("✅ Standard monitoring practices sufficient.")

## 9. Security Feature Analysis

Let's analyze the security features implemented across contracts.

In [ ]:
# Analyze security features across contracts
print("=== Security Features Analysis ===")

# Extract security features data
security_features_data = []

for contract_id, analysis in assessment.contract_analyses.items():
    features = analysis.security_features
    security_features_data.append({
        'Contract': analysis.contract_name.split()[0],
        'Access Control': features.access_control,
        'Reentrancy Protection': features.reentrancy_protection,
        'Integer Overflow Protection': features.integer_overflow_protection,
        'Emergency Pause': features.emergency_pause,
        'Timelock Mechanism': features.timelock_mechanism,
        'Multisig Requirement': features.multisig_requirement,
        'Input Validation': features.input_validation,
        'Event Logging': features.event_logging
    })

# Create DataFrame
features_df = pd.DataFrame(security_features_data)
features_df = features_df.set_index('Contract')

# Convert boolean to int for visualization
features_numeric = features_df.astype(int)

print("Security Features Implementation Matrix:")
print(features_df.to_string())

# Calculate security score for each contract
security_scores = features_numeric.sum(axis=1) / len(features_numeric.columns)
print(f"\nSecurity Implementation Scores:")
for contract, score in security_scores.items():
    grade = "A" if score >= 0.8 else "B" if score >= 0.6 else "C" if score >= 0.4 else "D"
    print(f"  {contract}: {score:.2f} ({grade})")

In [ ]:
# Visualize security features
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 8))
fig.suptitle('Security Features Analysis', fontsize=16, fontweight='bold')

# 1. Security Features Heatmap
im1 = ax1.imshow(features_numeric.T, cmap='RdYlGn', aspect='auto')
ax1.set_xticks(range(len(features_numeric.index)))
ax1.set_yticks(range(len(features_numeric.columns)))
ax1.set_xticklabels(features_numeric.index, rotation=45, ha='right')
ax1.set_yticklabels(features_numeric.columns)
ax1.set_title('Security Features Implementation\n(Green = Implemented, Red = Missing)')

# Add text annotations
for i in range(len(features_numeric.columns)):
    for j in range(len(features_numeric.index)):
        text = ax1.text(j, i, '✓' if features_numeric.iloc[j, i] else '✗',
                       ha="center", va="center", 
                       color="white" if features_numeric.iloc[j, i] else "red",
                       fontsize=12, fontweight='bold')

# 2. Security Score Distribution
colors = ['green' if score >= 0.8 else 'orange' if score >= 0.6 else 'red' for score in security_scores]
bars2 = ax2.bar(security_scores.index, security_scores.values, color=colors, alpha=0.7)
ax2.set_title('Overall Security Implementation Score')
ax2.set_ylabel('Security Score (0-1)')
ax2.set_ylim(0, 1)
ax2.tick_params(axis='x', rotation=45)

# Add score labels
for bar, score in zip(bars2, security_scores.values):
    grade = "A" if score >= 0.8 else "B" if score >= 0.6 else "C" if score >= 0.4 else "D"
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02, 
             f'{score:.2f}\n({grade})', ha='center', va='bottom', fontweight='bold')

plt.tight_layout()
plt.show()

## 10. Audit Analysis

Let's examine the audit history and quality across contracts.

In [ ]:
# Analyze audit information
print("=== Contract Audit Analysis ===")

audit_summary = []

for contract_id, analysis in assessment.contract_analyses.items():
    audits = analysis.audits
    
    if audits:
        latest_audit = max(audits, key=lambda x: x.audit_date)
        audit_count = len(audits)
        avg_score = sum(audit.audit_score for audit in audits) / len(audits)
        days_since_latest = (datetime.now() - latest_audit.audit_date).days
        
        total_vulnerabilities = sum(audit.vulnerabilities_found for audit in audits)
        total_fixed = sum(audit.vulnerabilities_fixed for audit in audits)
        fix_rate = (total_fixed / total_vulnerabilities * 100) if total_vulnerabilities > 0 else 100
        
        audit_summary.append({
            'Contract': analysis.contract_name,
            'Audit Count': audit_count,
            'Average Score': avg_score,
            'Latest Auditor': latest_audit.auditor,
            'Days Since Latest': days_since_latest,
            'Total Vulnerabilities': total_vulnerabilities,
            'Vulnerabilities Fixed': total_fixed,
            'Fix Rate %': fix_rate
        })
        
        print(f"\n{analysis.contract_name}:")
        print(f"  📊 Audits: {audit_count}")
        print(f"  🎯 Average Score: {avg_score:.1f}/100")
        print(f"  🔍 Latest Auditor: {latest_audit.auditor}")
        print(f"  📅 Days Since Latest: {days_since_latest}")
        print(f"  🐛 Vulnerabilities: {total_vulnerabilities} found, {total_fixed} fixed ({fix_rate:.1f}%)")
        
        # Audit history
        print(f"  📋 Audit History:")
        for audit in sorted(audits, key=lambda x: x.audit_date, reverse=True):
            print(f"    • {audit.audit_date.strftime('%Y-%m-%d')} - {audit.auditor} ({audit.audit_type})")
            print(f"      Score: {audit.audit_score}, Vulns: {audit.vulnerabilities_found}/{audit.vulnerabilities_fixed}")
    
    else:
        audit_summary.append({
            'Contract': analysis.contract_name,
            'Audit Count': 0,
            'Average Score': 0,
            'Latest Auditor': 'None',
            'Days Since Latest': 9999,
            'Total Vulnerabilities': 0,
            'Vulnerabilities Fixed': 0,
            'Fix Rate %': 0
        })
        
        print(f"\n{analysis.contract_name}:")
        print(f"  ❌ No audits conducted - HIGH RISK")

# Create audit summary DataFrame
audit_df = pd.DataFrame(audit_summary)
print(f"\n=== Audit Summary Table ===")
print(audit_df.to_string(index=False))

In [ ]:
# Visualize audit analysis
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
fig.suptitle('Contract Audit Analysis Dashboard', fontsize=16, fontweight='bold')

# 1. Audit Scores vs Count
ax1 = axes[0, 0]
audited_contracts = audit_df[audit_df['Audit Count'] > 0]
if not audited_contracts.empty:
    scatter = ax1.scatter(audited_contracts['Audit Count'], audited_contracts['Average Score'],
                         s=audited_contracts['Fix Rate %']*3, alpha=0.6, 
                         c=audited_contracts['Days Since Latest'], cmap='RdYlGn_r')
    ax1.set_xlabel('Number of Audits')
    ax1.set_ylabel('Average Audit Score')
    ax1.set_title('Audit Quality vs Quantity\n(Size = Fix Rate, Color = Days Since Latest)')
    
    # Add contract labels
    for i, row in audited_contracts.iterrows():
        ax1.annotate(row['Contract'].split()[0], 
                    (row['Audit Count'], row['Average Score']),
                    xytext=(5, 5), textcoords='offset points', fontsize=8)

# 2. Audit Coverage
ax2 = axes[0, 1]
audited_count = len(audit_df[audit_df['Audit Count'] > 0])
unaudited_count = len(audit_df[audit_df['Audit Count'] == 0])
labels = ['Audited', 'Unaudited']
sizes = [audited_count, unaudited_count]
colors = ['green', 'red']
explode = (0, 0.1)  # Explode unaudited slice

ax2.pie(sizes, explode=explode, labels=labels, colors=colors, autopct='%1.1f%%',
        shadow=True, startangle=90)
ax2.set_title('Audit Coverage Distribution')

# 3. Vulnerability Fix Rates
ax3 = axes[1, 0]
fix_rate_data = audit_df[audit_df['Total Vulnerabilities'] > 0]
if not fix_rate_data.empty:
    colors3 = ['green' if rate >= 90 else 'orange' if rate >= 70 else 'red' 
              for rate in fix_rate_data['Fix Rate %']]
    bars3 = ax3.bar(range(len(fix_rate_data)), fix_rate_data['Fix Rate %'], 
                    color=colors3, alpha=0.7)
    ax3.set_title('Vulnerability Fix Rates')
    ax3.set_ylabel('Fix Rate (%)')
    ax3.set_xticks(range(len(fix_rate_data)))
    ax3.set_xticklabels([name.split()[0] for name in fix_rate_data['Contract']], 
                       rotation=45, ha='right')
    ax3.set_ylim(0, 100)
    
    # Add percentage labels
    for bar, rate in zip(bars3, fix_rate_data['Fix Rate %']):
        ax3.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1, 
                f'{rate:.1f}%', ha='center', va='bottom', fontweight='bold')

# 4. Audit Freshness
ax4 = axes[1, 1]
fresh_audits = len(audit_df[(audit_df['Days Since Latest'] <= 365) & (audit_df['Audit Count'] > 0)])
stale_audits = len(audit_df[(audit_df['Days Since Latest'] > 365) & (audit_df['Audit Count'] > 0)])
no_audits = len(audit_df[audit_df['Audit Count'] == 0])

categories = ['Fresh\n(≤1 year)', 'Stale\n(>1 year)', 'No Audits']
values = [fresh_audits, stale_audits, no_audits]
colors4 = ['green', 'orange', 'red']

bars4 = ax4.bar(categories, values, color=colors4, alpha=0.7)
ax4.set_title('Audit Freshness Distribution')
ax4.set_ylabel('Number of Contracts')

# Add count labels
for bar, value in zip(bars4, values):
    ax4.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.05, 
             str(value), ha='center', va='bottom', fontweight='bold', fontsize=12)

plt.tight_layout()
plt.show()

## 11. Export Assessment Report

Let's export a comprehensive assessment report.

In [ ]:
# Export comprehensive assessment report
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
report_filename = f"smart_contract_risk_assessment_{timestamp}.json"

report_path = await contract_engine.export_assessment_report(assessment, report_filename)

print(f"📄 Comprehensive assessment report exported to: {report_path}")
print(f"📊 Report includes:")
print(f"  • Portfolio metrics and overall risk scores")
print(f"  • Individual contract security analyses")
print(f"  • Interaction risk assessments")
print(f"  • Exploit proximity analysis")
print(f"  • Risk alerts and recommendations")
print(f"  • Monitoring requirements")

## 12. Summary and Key Insights

Let's summarize the key insights from our smart contract risk analysis.

In [ ]:
# Generate comprehensive summary
print("=== Smart Contract Risk Assessment Summary ===")
print()

# Overall portfolio health
print(f"🏥 Portfolio Health Overview:")
print(f"  Overall Risk Score: {assessment.overall_portfolio_risk:.2f} ({assessment.risk_level})")
print(f"  Confidence Level: {assessment.confidence_score:.2f}")
print(f"  Next Review Date: {assessment.next_review_date.strftime('%Y-%m-%d')}")
print(f"  Total Contracts Analyzed: {assessment.portfolio_metrics.total_contracts}")

# Risk distribution
risk_tiers = {}
for score in assessment.contract_scores:
    risk_tiers[score.risk_tier] = risk_tiers.get(score.risk_tier, 0) + 1

print(f"\n📊 Risk Distribution:")
for tier, count in sorted(risk_tiers.items()):
    emoji = {"MINIMAL": "🟢", "LOW": "🔵", "MODERATE": "🟡", "HIGH": "🟠", "CRITICAL": "🔴"}.get(tier, "⚪")
    print(f"  {emoji} {tier}: {count} contract(s)")

# Top risk factors
print(f"\n⚠️ Key Risk Factors Identified:")
risk_factors = [
    f"Unaudited contracts: {assessment.portfolio_metrics.total_contracts - assessment.portfolio_metrics.audited_contracts}",
    f"Critical vulnerabilities: {assessment.portfolio_metrics.critical_vulnerabilities}",
    f"High-risk interactions: {assessment.portfolio_metrics.complex_interactions}",
    f"Admin key risks: {assessment.portfolio_metrics.admin_key_risks}",
    f"Upgrade risks: {assessment.portfolio_metrics.upgrade_risks}"
]

for factor in risk_factors:
    print(f"  • {factor}")

# Best and worst contracts
best_contract = min(assessment.contract_scores, key=lambda x: x.overall_risk_score)
worst_contract = max(assessment.contract_scores, key=lambda x: x.overall_risk_score)

print(f"\n🏆 Security Champions:")
print(f"  🥇 Best: {best_contract.contract_name} (Risk: {best_contract.overall_risk_score:.2f})")
print(f"  🥉 Needs Attention: {worst_contract.contract_name} (Risk: {worst_contract.overall_risk_score:.2f})")

# Priority actions
critical_alerts = [alert for alert in assessment.risk_alerts if alert.severity == 'CRITICAL']
high_alerts = [alert for alert in assessment.risk_alerts if alert.severity == 'HIGH']

print(f"\n🚨 Priority Actions Required:")
print(f"  Critical Alerts: {len(critical_alerts)}")
print(f"  High Priority Alerts: {len(high_alerts)}")

if critical_alerts or high_alerts:
    print(f"  ⚡ Immediate Actions:")
    for alert in (critical_alerts + high_alerts)[:3]:  # Top 3 urgent
        print(f"    • {alert.message}")

# Security best practices compliance
total_features = len(security_features_data[0]) - 1  # Exclude contract name
avg_implementation = security_scores.mean()

print(f"\n🛡️ Security Best Practices Compliance:")
print(f"  Average Implementation: {avg_implementation:.1%}")
compliance_grade = "A" if avg_implementation >= 0.8 else "B" if avg_implementation >= 0.6 else "C" if avg_implementation >= 0.4 else "D"
print(f"  Portfolio Grade: {compliance_grade}")

# Recommendations summary
print(f"\n💡 Strategic Recommendations:")
strategic_recommendations = [
    "Prioritize security audits for unaudited contracts",
    "Implement missing security features (timelock, multisig, emergency pause)",
    "Establish continuous security monitoring processes",
    "Create incident response procedures for security events",
    "Regular security training for development teams",
    "Implement automated security testing in CI/CD pipeline"
]

for i, rec in enumerate(strategic_recommendations, 1):
    print(f"  {i}. ✅ {rec}")

# Risk management framework
print(f"\n🎯 Comprehensive Risk Management Framework:")
framework_components = [
    "Continuous Monitoring: Real-time contract and interaction monitoring",
    "Proactive Auditing: Regular security audits and code reviews",
    "Incident Response: Rapid response procedures for security events",
    "Risk Assessment: Periodic comprehensive risk evaluations",
    "Security Standards: Enforcement of security best practices",
    "Vulnerability Management: Systematic tracking and remediation"
]

for i, component in enumerate(framework_components, 1):
    print(f"  {i}. 🛡️ {component}")

print("\n" + "="*70)
print("Smart Contract Risk Assessment Demo Completed Successfully!")
print("="*70)

# Final risk score interpretation
risk_interpretation = {
    "CRITICAL": "Immediate action required - high security risks detected",
    "HIGH": "Significant risks identified - prompt attention needed",
    "MODERATE": "Some risks present - regular monitoring recommended",
    "LOW": "Good security posture - maintain current practices",
    "MINIMAL": "Excellent security - continue best practices"
}

print(f"\n🎯 Risk Level Interpretation:")
print(f"   {assessment.risk_level}: {risk_interpretation.get(assessment.risk_level, 'Unknown risk level')}")